In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import log_loss, accuracy_score
from scipy.sparse import hstack


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train = pd.read_csv('/kaggle/input/llm-classification-finetuning/train.csv')
test = pd.read_csv('/kaggle/input/llm-classification-finetuning/test.csv')

In [ ]:
def map_winner(row):
    if row.winner_model_a == 1: return 'A'
    if row.winner_model_b == 1: return 'B'
    return 'tie'

train['winner'] = train.apply(map_winner, axis=1)

In [ ]:
label_map = {'A':0, 'B':1, 'tie':2}
y = train['winner'].map(label_map)

# 4) Model A: full-text TF–IDF + Logistic Regression
print("\n=== Model A: Full-text TF–IDF ===")
full_text = (
    train['prompt'] + ' [SEP] ' +
    train['response_a'] + ' [VS] ' +
    train['response_b']
)
vectorizer_A = TfidfVectorizer(max_features=20000, ngram_range=(1,2), strip_accents='unicode')
X_A = vectorizer_A.fit_transform(full_text)
print("X_A shape:", X_A.shape)

clf_A = LogisticRegression(multi_class='multinomial', solver='saga', C=1.0, max_iter=500)

print("Cross-validating Model A...")
cv = 5
scores_ll_A = cross_val_score(clf_A, X_A, y, cv=cv, scoring='neg_log_loss', n_jobs=-1)
scores_acc_A = cross_val_score(clf_A, X_A, y, cv=cv, scoring='accuracy', n_jobs=-1)
print(f"Model A CV log-loss: {-scores_ll_A.mean():.4f} ± {scores_ll_A.std():.4f}")
print(f"Model A CV accuracy: {scores_acc_A.mean():.4f} ± {scores_acc_A.std():.4f}")

# 5) Model B: separate prompt & response TF–IDF + difference
print("\n=== Model B: Separate TF–IDF + Difference ===")
vectorizer_prompt = TfidfVectorizer(max_features=10000, ngram_range=(1,2), strip_accents='unicode')
vectorizer_resp = TfidfVectorizer(max_features=10000, ngram_range=(1,2), strip_accents='unicode')

X_prompt = vectorizer_prompt.fit_transform(train['prompt'])
X_respA = vectorizer_resp.fit_transform(train['response_a'])
X_respB = vectorizer_resp.transform(train['response_b'])
print("Shapes:", X_prompt.shape, X_respA.shape, X_respB.shape)

X_diff = X_respA - X_respB
X_B = hstack([X_prompt, X_respA, X_respB, X_diff])
print("X_B shape:", X_B.shape)

clf_B = LogisticRegression(multi_class='multinomial', solver='saga', C=1.0, max_iter=500)

print("Cross-validating Model B...")
scores_ll_B = cross_val_score(clf_B, X_B, y, cv=cv, scoring='neg_log_loss', n_jobs=-1)
scores_acc_B = cross_val_score(clf_B, X_B, y, cv=cv, scoring='accuracy', n_jobs=-1)
print(f"Model B CV log-loss: {-scores_ll_B.mean():.4f} ± {scores_ll_B.std():.4f}")
print(f"Model B CV accuracy: {scores_acc_B.mean():.4f} ± {scores_acc_B.std():.4f}")

# 6) Train both on full data
print("\nTraining Model A on full training set...")
clf_A.fit(X_A, y)
print("Training Model B on full training set...")
clf_B.fit(X_B, y)

# 7) Transform test data
print("\nTransforming test data...")
full_text_test = (
    test['prompt'] + ' [SEP] ' +
    test['response_a'] + ' [VS] ' +
    test['response_b']
)
X_A_test = vectorizer_A.transform(full_text_test)

X_prompt_test = vectorizer_prompt.transform(test['prompt'])
X_respA_test = vectorizer_resp.transform(test['response_a'])
X_respB_test = vectorizer_resp.transform(test['response_b'])
X_diff_test = X_respA_test - X_respB_test
X_B_test = hstack([X_prompt_test, X_respA_test, X_respB_test, X_diff_test])

# 8) Predict probabilities
print("Predicting with Model A...")
probs_A = clf_A.predict_proba(X_A_test)
print("Predicting with Model B...")
probs_B = clf_B.predict_proba(X_B_test)

# 9) Ensemble by averaging
print("\nEnsembling predictions...")
probs_ensemble = (probs_A + probs_B) / 2

# 10) Build and save submission
print("Building submission file...")
submission = pd.DataFrame({
    'id': test['id'],
    'winner_model_a': probs_ensemble[:, label_map['A']],
    'winner_model_b': probs_ensemble[:, label_map['B']],
    'winner_model_tied': probs_ensemble[:, label_map['tie']],
})
submission.to_csv('submission.csv', index=False)
print("Saved submission.csv")